In [11]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession
from main import load_trips, load_zones
from m1 import flag_invalid, clean

In [12]:
spark = (
    SparkSession.builder
    .appName("ReportMetrics")
    .master("local[*]")
    .getOrCreate()
)

In [14]:
print("trips" in globals())
print([x for x in globals() if x in ["spark", "trips", "years", "zones"]])

False
['spark', 'years']


In [16]:
trips = load_trips(
    spark,
    "../databases",
    years
)

In [17]:
flagged = flag_invalid(trips, years)
trips_clean = clean(flagged)

In [19]:
years = [2025]

trips = load_trips(
    spark,
    "../databases",
    years
)

zones = load_zones(
    spark,
    "../databases/taxi_zone_lookup.csv"
)

In [20]:
print("Raw trips:", trips.count())
print("Cleaned trips:", trips_clean.count())
print("Zones:", zones.count())


Raw trips: 19348004
Cleaned trips: 17483653
Zones: 265


In [23]:
import glob
import os

files = glob.glob("../databases/yellow_tripdata_2025-*.parquet")

total_bytes = sum(os.path.getsize(f) for f in files)
total_gb = total_bytes / (1024 ** 3)

print("Files:", len(files))
print(f"Total input size: {total_gb:.2f} GB")

Files: 5
Total input size: 0.31 GB


In [27]:
from m1 import cleaning_stats
stats = cleaning_stats(flagged)
stats.orderBy("year").show(truncate=False)

+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|year|total   |removed_fare|removed_dist|removed_pax|removed_time|removed_zone|removed_period|removed_null|removed_any|
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|2007|1       |0           |0           |0          |0           |0           |1             |0           |1          |
|2009|2       |0           |0           |0          |0           |0           |2             |0           |2          |
|2024|21      |0           |0           |0          |0           |0           |21            |0           |21         |
|2025|19347980|1042872     |542643      |111923     |144203      |145768      |0             |0           |1864327    |
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+

